# UNHD Part A -- Tesseract Baseline
Runs Tesseract (with Urdu language pack) over the evaluation set, computes CER/WER (diacritic-sensitive and insensitive), and produces per-sentence predictions plus error analysis.

## Cell 1 -- Install Tesseract + Urdu language pack

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-urd
!pip install -q pytesseract python-Levenshtein

import pytesseract
print(pytesseract.get_tesseract_version())
print("urd" in pytesseract.get_languages())

## Cell 2 -- Locate dataset automatically

In [ ]:
import os

def find_dataset_root():
    for root, dirs, files in os.walk("/kaggle/input"):
        if "evaluation_set.csv" in files:
            return root
    raise FileNotFoundError("Could not find evaluation_set.csv anywhere under /kaggle/input")

DATASET_ROOT = find_dataset_root()
CSV_PATH = os.path.join(DATASET_ROOT, "evaluation_set.csv")
IMAGES_DIR = os.path.join(DATASET_ROOT, "images")

print("CSV:", CSV_PATH)
print("Images dir:", IMAGES_DIR)
print("Images found:", len(os.listdir(IMAGES_DIR)))

## Cell 3 -- Load evaluation set, sanity-check against images on disk

In [ ]:
import csv

with open(CSV_PATH, encoding="utf-8-sig") as f:
    eval_rows = list(csv.DictReader(f))

print(f"Evaluation set rows: {len(eval_rows)}")

def image_filename(row):
    # normalize backslashes regardless of what OS wrote the path
    return row["image_path"].replace("\\", "/").rsplit("/", 1)[-1]

missing = [r for r in eval_rows if not os.path.isfile(os.path.join(IMAGES_DIR, image_filename(r)))]
print(f"Missing images: {len(missing)}")
if missing:
    print("Examples:", [image_filename(r) for r in missing[:5]])
    print("STOP -- investigate before running the full pipeline.")

## Cell 4 -- CER/WER functions (matches proposal Section 6.1 definition)

In [ ]:
import unicodedata

def edit_distance(a, b):
    """Standard Levenshtein edit distance, sequence of any hashable units."""
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if a[i-1] == b[j-1] else 1
            curr[j] = min(
                prev[j] + 1,        # deletion
                curr[j-1] + 1,      # insertion
                prev[j-1] + cost    # substitution
            )
        prev = curr
    return prev[m]

def strip_diacritics(text):
    """Removes Unicode combining marks (category 'Mn') -- covers Urdu/Arabic
    i'raab (zabar, zer, pesh, shadda, sukun, tanween, etc.)."""
    normalized = unicodedata.normalize("NFC", text)
    return "".join(ch for ch in normalized if unicodedata.category(ch) != "Mn")

def clean_text(text):
    return unicodedata.normalize("NFC", (text or "").strip())

def cer(pred, gt):
    gt = clean_text(gt)
    pred = clean_text(pred)
    if len(gt) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, gt) / len(gt)

def wer(pred, gt):
    gt_words = clean_text(gt).split()
    pred_words = clean_text(pred).split()
    if len(gt_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return edit_distance(pred_words, gt_words) / len(gt_words)

def compute_metrics(pred, gt):
    """Returns (CER_sensitive, CER_insensitive, WER_sensitive, WER_insensitive)."""
    cer_sens = cer(pred, gt)
    wer_sens = wer(pred, gt)
    pred_stripped = strip_diacritics(pred)
    gt_stripped = strip_diacritics(gt)
    cer_insens = cer(pred_stripped, gt_stripped)
    wer_insens = wer(pred_stripped, gt_stripped)
    return cer_sens, cer_insens, wer_sens, wer_insens

# quick self-test
assert edit_distance("kitten", "sitting") == 3
print("Metric functions OK")

## Cell 5 -- Run Tesseract with checkpointing (safe to interrupt and resume)
Set `PILOT_SIZE = 100` first to sanity-check, then set to `None` and rerun to do the full 2,465 (it will skip what's already done).

In [ ]:
import pytesseract
from PIL import Image
from tqdm.notebook import tqdm

PILOT_SIZE = 100   # set to None to run the full evaluation set
OUTPUT_CSV = "/kaggle/working/tesseract_predictions.csv"

rows_to_run = eval_rows if PILOT_SIZE is None else eval_rows[:PILOT_SIZE]

# resume support: skip file_ids already done in a previous run
done_ids = set()
if os.path.isfile(OUTPUT_CSV):
    with open(OUTPUT_CSV, encoding="utf-8") as f:
        done_ids = {r["file_id"] for r in csv.DictReader(f)}
    print(f"Resuming -- {len(done_ids)} already done, skipping those")

fieldnames = ["sentence_id", "file_id", "writer_id", "ground_truth", "prediction",
              "cer_sensitive", "cer_insensitive", "wer_sensitive", "wer_insensitive", "error"]

write_header = not os.path.isfile(OUTPUT_CSV)
out_f = open(OUTPUT_CSV, "a", newline="", encoding="utf-8")
writer = csv.DictWriter(out_f, fieldnames=fieldnames)
if write_header:
    writer.writeheader()

CHECKPOINT_EVERY = 25
processed_since_flush = 0

for row in tqdm(rows_to_run, desc="Tesseract OCR"):
    if row["file_id"] in done_ids:
        continue

    img_path = os.path.join(IMAGES_DIR, image_filename(row))
    try:
        img = Image.open(img_path)
        prediction = pytesseract.image_to_string(img, lang="urd").strip()
        cer_s, cer_i, wer_s, wer_i = compute_metrics(prediction, row["text"])
        error_msg = ""
    except Exception as e:
        prediction = ""
        cer_s = cer_i = wer_s = wer_i = 1.0
        error_msg = str(e)

    writer.writerow({
        "sentence_id": row["sentence_id"],
        "file_id": row["file_id"],
        "writer_id": row["writer_id"],
        "ground_truth": row["text"],
        "prediction": prediction,
        "cer_sensitive": cer_s,
        "cer_insensitive": cer_i,
        "wer_sensitive": wer_s,
        "wer_insensitive": wer_i,
        "error": error_msg,
    })

    processed_since_flush += 1
    if processed_since_flush >= CHECKPOINT_EVERY:
        out_f.flush()
        processed_since_flush = 0

out_f.close()
print(f"\nDone. Predictions saved to {OUTPUT_CSV}")

## Cell 6 -- Aggregate results

In [ ]:
import pandas as pd

results = pd.read_csv(OUTPUT_CSV)
results["error"] = results["error"].fillna("")  # blank CSV cells become NaN on read -- normalize back to ""
results["prediction"] = results["prediction"].fillna("")
print(f"Total predictions: {len(results)}")
print(f"Failed/errored images: {(results['error'] != '').sum()}")

print("\n--- AGGREGATE METRICS (mean across all sentences) ---")
print(f"CER (diacritic-sensitive):   {results['cer_sensitive'].mean():.4f}")
print(f"CER (diacritic-insensitive): {results['cer_insensitive'].mean():.4f}")
print(f"WER (diacritic-sensitive):   {results['wer_sensitive'].mean():.4f}")
print(f"WER (diacritic-insensitive): {results['wer_insensitive'].mean():.4f}")

print(f"\nSentences with 0% CER (perfect match): {(results['cer_sensitive']==0).sum()} / {len(results)}")
print(f"Sentences with 100%+ CER (worse than blank): {(results['cer_sensitive']>=1.0).sum()} / {len(results)}")
print(f"Empty predictions (Tesseract found nothing): {(results['prediction'].fillna('')=='').sum()} / {len(results)}")

## Cell 7 -- Error analysis: actual sentence vs prediction, worst/best cases, diagnosis

In [ ]:
def show_examples(df, title, n=5):
    print(f"\n{'='*80}\n{title}\n{'='*80}")
    for _, row in df.head(n).iterrows():
        print(f"\nfile_id: {row['file_id']}  |  CER: {row['cer_sensitive']:.2f}  |  WER: {row['wer_sensitive']:.2f}")
        print(f"  GROUND TRUTH: {row['ground_truth']}")
        pred_display = row['prediction'] if row['prediction'] else '[EMPTY -- Tesseract returned nothing]'
        print(f"  PREDICTION:   {pred_display}")

worst = results.sort_values("cer_sensitive", ascending=False)
best = results.sort_values("cer_sensitive", ascending=True)

show_examples(worst, "WORST 5 PREDICTIONS (highest CER)")
show_examples(best[best["cer_sensitive"] > 0], "BEST NON-PERFECT 5 PREDICTIONS", n=5)

# Diagnose *why* it's struggling
print(f"\n{'='*80}\nWHY IS IT STRUGGLING -- automatic diagnosis\n{'='*80}")
empty_pred = (results["prediction"].fillna("") == "")
print(f"- {empty_pred.sum()} / {len(results)} predictions were completely empty.")
print(f"  This usually means Tesseract's segmentation failed on Nastaliq's cursive,")
print(f"  connected letterforms entirely -- not a recognition-quality issue but a")
print(f"  script-support gap (Tesseract's Urdu model is tuned closer to Naskh/printed text).")

diacritic_only_error = ((results["cer_sensitive"] > 0) & (results["cer_insensitive"] < results["cer_sensitive"] * 0.5))
print(f"\n- {diacritic_only_error.sum()} / {len(results)} sentences see CER drop by >50% once diacritics")
print(f"  are ignored -- for these, Tesseract is getting the base letters roughly right")
print(f"  but losing the i'raab marks specifically.")

near_total_failure = (results["cer_sensitive"] >= 0.9)
print(f"\n- {near_total_failure.sum()} / {len(results)} sentences have CER >= 0.9 (near-total failure,")
print(f"  i.e. almost none of the ground truth characters were recovered at all).")

## Cell 8 -- Save a clean summary report

In [ ]:
summary_path = "/kaggle/working/tesseract_summary.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("TESSERACT BASELINE -- RESULTS SUMMARY\n")
    f.write("="*50 + "\n")
    f.write(f"Images evaluated: {len(results)}\n")
    f.write(f"CER (diacritic-sensitive):   {results['cer_sensitive'].mean():.4f}\n")
    f.write(f"CER (diacritic-insensitive): {results['cer_insensitive'].mean():.4f}\n")
    f.write(f"WER (diacritic-sensitive):   {results['wer_sensitive'].mean():.4f}\n")
    f.write(f"WER (diacritic-insensitive): {results['wer_insensitive'].mean():.4f}\n")
    f.write(f"Empty predictions: {(results['prediction'].fillna('')=='').sum()} / {len(results)}\n")

print(f"Saved: {summary_path}")
print(f"Predictions CSV: {OUTPUT_CSV}")